# 04 – Load the clean data into a SQL database
Creates a SQLite database (`data/processed/complaints.db`) with one table, `complaints`, so the analysis can be done in SQL.
SQLite is built into Python – nothing to install.

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/processed/complaints_clean.csv", parse_dates=["date_received", "date_sent"])

# store dates as YYYY-MM-DD text (the standard way in SQLite) and flags as 0/1
for col in ["date_received", "date_sent"]:
    df[col] = df[col].dt.strftime("%Y-%m-%d")
for col in ["is_closed", "got_relief", "got_monetary_relief", "is_timely"]:
    df[col] = df[col].astype(int)

print(f"{len(df):,} rows ready to load")

511,395 rows ready to load


In [2]:
conn = sqlite3.connect("../data/processed/complaints.db")

df.to_sql("complaints", conn, if_exists="replace", index=False)

# indexes make filtering and grouping by these columns much faster
conn.execute("CREATE INDEX IF NOT EXISTS idx_company ON complaints(company)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_product ON complaints(product)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_year_month ON complaints(year_month)")
conn.commit()
print("Database created: ../data/processed/complaints.db")

Database created: ../data/processed/complaints.db


## Test queries
A helper function runs any SQL and shows the result as a table.

In [3]:
def q(sql):
    return pd.read_sql(sql, conn)

q("SELECT COUNT(*) AS total_complaints FROM complaints")

,total_complaints
0,511395


In [4]:
q("""
SELECT product,
       COUNT(*) AS complaints
FROM complaints
GROUP BY product
ORDER BY complaints DESC
""")

,product,complaints
0,Checking or savings account,264653
1,Credit card,246742


In [5]:
q("""
SELECT ROUND(100.0 * SUM(got_relief) / COUNT(*), 1)          AS relief_rate_pct,
       ROUND(100.0 * SUM(got_monetary_relief) / COUNT(*), 1) AS monetary_relief_rate_pct
FROM complaints
WHERE is_closed = 1
""")

,relief_rate_pct,monetary_relief_rate_pct
0,23.7,15.2
